# Backtracking — LeetCode Questions (8 Qs)

Backtracking = DFS + undo choice if it leads to dead end.

Template:
```
def backtrack(start, path):
    if base_case:
        result.append(path[:])
        return
    for choice in choices:
        path.append(choice)       # make choice
        backtrack(next, path)     # explore
        path.pop()                # undo choice (BACKTRACK)
```

1.  78 – Subsets
2.  90 – Subsets II (with duplicates)
3.  46 – Permutations
4.  47 – Permutations II (with duplicates)
5.  39 – Combination Sum
6.  40 – Combination Sum II
7.  17 – Letter Combinations of a Phone Number
8.  79 – Word Search

In [ ]:
from typing import List

# 1. LeetCode 78 – Subsets

Return all possible subsets (power set). No duplicates. Order doesn't matter.

Input: [1,2,3]  Output: [[],[1],[2],[3],[1,2],[1,3],[2,3],[1,2,3]]

## Dry Run

Backtrack: at each index, decide include or skip.

| call | path | start | action |
|------|------|-------|--------|
| bt(0, []) | [] | 0 | add [], try 1,2,3 |
| bt(1, [1]) | [1] | 1 | add [1], try 2,3 |
| bt(2, [1,2]) | [1,2] | 2 | add [1,2], try 3 |
| bt(3, [1,2,3]) | [1,2,3] | 3 | add [1,2,3], end |
| backtrack | [1,2] → [1] | | remove 3 |
| bt(3, [1,3]) | [1,3] | 3 | add [1,3], end |
| ... and so on | | | |

In [ ]:
class Solution:
    def subsets(self, nums: List[int]) -> List[List[int]]:
        result = []

        def backtrack(start, path):
            result.append(path[:])      # add current subset (including empty)
            for i in range(start, len(nums)):
                path.append(nums[i])
                backtrack(i + 1, path)
                path.pop()              # undo choice

        backtrack(0, [])
        return result

print(Solution().subsets([1,2,3]))  # 8 subsets

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n * 2^n) – 2^n subsets, each takes O(n) to copy | O(n) – recursion depth and path |
| **Final: O(n * 2^n)** | **Final: O(n)** |

# 2. LeetCode 90 – Subsets II (with duplicates)

Like Subsets but input may have duplicates. Return unique subsets only.

Input: [1,2,2]  Output: [[],[1],[1,2],[1,2,2],[2],[2,2]]

## Dry Run

Sort first! Skip duplicates at same recursion level.

Sorted: [1,2,2]

| call | path | skip condition |
|------|------|----------------|
| bt(0,[]) | [] | add [] |
| i=0 | [1] → explore → backtrack | |
| i=1 | [2] → explore | |
| i=2 | i>start and nums[2]==nums[1] → **SKIP** | |
| So [2,2] found through [2] branch, not starting fresh from second 2 | | |

In [ ]:
class Solution:
    def subsetsWithDup(self, nums: List[int]) -> List[List[int]]:
        nums.sort()
        result = []

        def backtrack(start, path):
            result.append(path[:])
            for i in range(start, len(nums)):
                if i > start and nums[i] == nums[i-1]:
                    continue              # skip duplicate at same level
                path.append(nums[i])
                backtrack(i + 1, path)
                path.pop()

        backtrack(0, [])
        return result

print(Solution().subsetsWithDup([1,2,2]))  # [[], [1], [1,2], [1,2,2], [2], [2,2]]

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n * 2^n) worst case | O(n) – recursion depth |
| **Final: O(n * 2^n)** | **Final: O(n)** |

# 3. LeetCode 46 – Permutations

Return all permutations of distinct integers.

Input: [1,2,3]  Output: [[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]

## Dry Run

Use 'used' array to track which elements are in current path.

| call | path | used | action |
|------|------|------|--------|
| bt([]) | [] | [F,F,F] | try 0,1,2 |
| bt([1]) | [1] | [T,F,F] | try 1,2 |
| bt([1,2]) | [1,2] | [T,T,F] | try 2 |
| bt([1,2,3]) | [1,2,3] | [T,T,T] | full → add, backtrack |
| bt([1,3]) | [1,3] | [T,F,T] | try 1 |
| bt([1,3,2]) | [1,3,2] | [T,T,T] | full → add |
| ... | | | 6 total |

In [ ]:
class Solution:
    def permute(self, nums: List[int]) -> List[List[int]]:
        result = []
        used = [False] * len(nums)

        def backtrack(path):
            if len(path) == len(nums):
                result.append(path[:])
                return
            for i in range(len(nums)):
                if used[i]:
                    continue
                used[i] = True
                path.append(nums[i])
                backtrack(path)
                path.pop()
                used[i] = False

        backtrack([])
        return result

print(Solution().permute([1,2,3]))  # 6 permutations

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n * n!) – n! permutations, each length n | O(n) – recursion depth + used array |
| **Final: O(n * n!)** | **Final: O(n)** |

# 4. LeetCode 47 – Permutations II (with duplicates)

Return all UNIQUE permutations (input may have duplicates).

Input: [1,1,2]  Output: [[1,1,2],[1,2,1],[2,1,1]]

## Dry Run

Sort first. Skip: if nums[i]==nums[i-1] AND used[i-1] is False.
(This means same value, same level, previous sibling already explored)

| skip rule | why |
|-----------|-----|
| nums[i]==nums[i-1] and not used[i-1] | The i-1 position already explored this value at this level; using i would duplicate |

In [ ]:
class Solution:
    def permuteUnique(self, nums: List[int]) -> List[List[int]]:
        nums.sort()
        result = []
        used = [False] * len(nums)

        def backtrack(path):
            if len(path) == len(nums):
                result.append(path[:])
                return
            for i in range(len(nums)):
                if used[i]:
                    continue
                if i > 0 and nums[i] == nums[i-1] and not used[i-1]:
                    continue       # skip duplicate
                used[i] = True
                path.append(nums[i])
                backtrack(path)
                path.pop()
                used[i] = False

        backtrack([])
        return result

print(Solution().permuteUnique([1,1,2]))  # [[1,1,2],[1,2,1],[2,1,1]]

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n * n!) worst case, better with duplicates | O(n) – recursion + used array |
| **Final: O(n * n!)** | **Final: O(n)** |

# 5. LeetCode 39 – Combination Sum

Find all combinations summing to target. Same element can be reused.

Input: candidates=[2,3,6,7], target=7  Output: [[2,2,3],[7]]

## Dry Run

| path | remaining | action |
|------|-----------|--------|
| [] | 7 | try 2,3,6,7 |
| [2] | 5 | try 2,3,6,7 (from 2 onwards, reuse allowed) |
| [2,2] | 3 | try 2,3 |
| [2,2,2] | 1 | try 2 → 1<2 done |
| [2,2,3] | 0 | **found! add [2,2,3]** |
| backtrack... | | |
| [7] | 0 | **found! add [7]** |

In [ ]:
class Solution:
    def combinationSum(self, candidates: List[int], target: int) -> List[List[int]]:
        result = []

        def backtrack(start, path, remaining):
            if remaining == 0:
                result.append(path[:])
                return
            for i in range(start, len(candidates)):
                if candidates[i] > remaining:
                    continue         # prune: too big
                path.append(candidates[i])
                backtrack(i, path, remaining - candidates[i])  # i not i+1 (reuse allowed)
                path.pop()

        backtrack(0, [], target)
        return result

print(Solution().combinationSum([2,3,6,7], 7))   # [[2,2,3],[7]]
print(Solution().combinationSum([2,3,5], 8))      # [[2,2,2,2],[2,3,3],[3,5]]

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n^(target/min)) – exponential but pruned | O(target/min) – recursion depth |
| **Final: Exponential (pruned in practice)** | **Final: O(target/min_candidate)** |

# 6. LeetCode 40 – Combination Sum II

Like Combination Sum but each number used ONCE, input may have duplicates.

Input: candidates=[10,1,2,7,6,1,5], target=8  Output: [[1,1,6],[1,2,5],[1,7],[2,6]]

## Dry Run

Sort: [1,1,2,5,6,7,10]

| path | note |
|------|------|
| [1,1,6] | use both 1s |
| [1,2,5] | use first 1, skip second 1 at same level |
| [1,7] | |
| [2,6] | |
| Skip rule: if i>start and nums[i]==nums[i-1]: skip | avoids duplicate combos |

In [ ]:
class Solution:
    def combinationSum2(self, candidates: List[int], target: int) -> List[List[int]]:
        candidates.sort()
        result = []

        def backtrack(start, path, remaining):
            if remaining == 0:
                result.append(path[:])
                return
            for i in range(start, len(candidates)):
                if candidates[i] > remaining:
                    break            # sorted, rest too big
                if i > start and candidates[i] == candidates[i-1]:
                    continue         # skip duplicate at same level
                path.append(candidates[i])
                backtrack(i + 1, path, remaining - candidates[i])  # i+1 (no reuse)
                path.pop()

        backtrack(0, [], target)
        return result

print(Solution().combinationSum2([10,1,2,7,6,1,5], 8))  # [[1,1,6],[1,2,5],[1,7],[2,6]]

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(2^n) – each element included or not | O(n) – recursion depth |
| **Final: O(2^n)** | **Final: O(n)** |

# 7. LeetCode 17 – Letter Combinations of a Phone Number

Return all possible letter combinations for digit string.

Input: digits='23'  Output: ['ad','ae','af','bd','be','bf','cd','ce','cf']

## Dry Run

Phone map: 2→'abc', 3→'def'

| call | path | digit | letters | tree |
|------|------|-------|---------|------|
| bt(0,'') | '' | '2' | abc | try a,b,c |
| bt(1,'a') | 'a' | '3' | def | try d,e,f → add 'ad','ae','af' |
| bt(1,'b') | 'b' | '3' | def | add 'bd','be','bf' |
| bt(1,'c') | 'c' | '3' | def | add 'cd','ce','cf' |

In [ ]:
class Solution:
    def letterCombinations(self, digits: str) -> List[str]:
        if not digits:
            return []

        phone = {
            '2': 'abc', '3': 'def', '4': 'ghi', '5': 'jkl',
            '6': 'mno', '7': 'pqrs', '8': 'tuv', '9': 'wxyz'
        }
        result = []

        def backtrack(index, path):
            if index == len(digits):
                result.append(''.join(path))
                return
            for char in phone[digits[index]]:
                path.append(char)
                backtrack(index + 1, path)
                path.pop()

        backtrack(0, [])
        return result

print(Solution().letterCombinations('23'))  # ['ad','ae','af','bd','be','bf','cd','ce','cf']
print(Solution().letterCombinations(''))    # []

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(4^n * n) – 4^n combos (max 4 letters per digit), each length n | O(n) – recursion depth |
| **Final: O(4^n * n)** | **Final: O(n)** |

# 8. LeetCode 79 – Word Search

DFS on grid to find if word exists as adjacent (up/down/left/right) path.
Each cell used at most once.

Input: board=[['A','B','C','E'],['S','F','C','S'],['A','D','E','E']], word='ABCCED'
Output: True

## Dry Run

Find 'A' at (0,0), DFS outward:
(0,0)A → (0,1)B → (0,2)C → (1,2)C → (2,2)E → (2,1)D ✓ 'ABCCED'

| step | pos | char | word[i] | match? |
|------|-----|------|---------|--------|
| 1 | (0,0) | A | A | yes, mark visited |
| 2 | (0,1) | B | B | yes |
| 3 | (0,2) | C | C | yes |
| 4 | (1,2) | C | C | yes |
| 5 | (2,2) | E | E | yes |
| 6 | (2,1) | D | D | yes → word found! |

In [ ]:
class Solution:
    def exist(self, board: List[List[str]], word: str) -> bool:
        rows, cols = len(board), len(board[0])

        def dfs(r, c, i):
            if i == len(word):
                return True
            if r < 0 or r >= rows or c < 0 or c >= cols:
                return False
            if board[r][c] != word[i]:
                return False
            temp = board[r][c]
            board[r][c] = '#'       # mark visited
            found = (dfs(r+1,c,i+1) or dfs(r-1,c,i+1) or
                     dfs(r,c+1,i+1) or dfs(r,c-1,i+1))
            board[r][c] = temp      # restore (backtrack)
            return found

        for r in range(rows):
            for c in range(cols):
                if dfs(r, c, 0):
                    return True
        return False

board = [['A','B','C','E'],['S','F','C','S'],['A','D','E','E']]
print(Solution().exist(board, 'ABCCED'))  # True
print(Solution().exist(board, 'ABCB'))    # False

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(m*n * 4^len(word)) – start at each cell, 4 choices per step | O(len(word)) – recursion depth |
| **Final: O(m*n * 4^L)** where L=word length | **Final: O(L)** |